# Insurance Customer 360: CRM, Policy & Claims Risk Analytics

This notebook analyses the IBM Watson Marketing Customer Value sample from a CRM, customer-value, policy and descriptive claims-risk perspective.

**Important:** `Response` is a marketing response field, not churn. The dataset has no fraud label, so the project does not claim fraud detection.

## 1. Load and validate the source data

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

RAW = Path('../data/raw/WA_Fn-UseC_-Marketing-Customer-Value-Analysis.csv')
df = pd.read_csv(RAW)
df.shape, df.isna().sum().sum(), df.duplicated().sum()

Expected source quality from the copy used to build this portfolio: **9,134 rows × 24 columns, 0 blank cells, 0 exact duplicate rows**.

## 2. Business KPIs

In [ ]:
kpis = {
    'Customers': len(df),
    'Response Rate': (df['Response'].eq('Yes')).mean(),
    'Average CLV': df['Customer Lifetime Value'].mean(),
    'Average Monthly Premium': df['Monthly Premium Auto'].mean(),
    'Average Claim': df['Total Claim Amount'].mean(),
    'Customers With Open Complaint': (df['Number of Open Complaints'].gt(0)).mean(),
}
pd.Series(kpis)

## 3. CRM: channel and offer response

In [ ]:
channel = (df.assign(response_flag=df['Response'].eq('Yes').astype(int))
             .groupby('Sales Channel')
             .agg(customers=('Customer','count'), response_rate=('response_flag','mean'), avg_clv=('Customer Lifetime Value','mean'))
             .sort_values('response_rate', ascending=False))
channel

In [ ]:
offer_channel = (df.assign(response_flag=df['Response'].eq('Yes').astype(int))
                 .groupby(['Renew Offer Type','Sales Channel'])
                 .agg(customers=('Customer','count'), response_rate=('response_flag','mean'))
                 .query('customers >= 50')
                 .sort_values(['response_rate','customers'], ascending=[False,False]))
offer_channel.head(10)

Observed response differences are **associations**, not causal effects. Offer assignment and channel selection are not randomised in this sample.

## 4. Customer value segmentation

In [ ]:
df['CLV Tier'] = pd.qcut(df['Customer Lifetime Value'], 4, labels=['Low','Mid-Low','Mid-High','High'])
value_summary = (df.assign(response_flag=df['Response'].eq('Yes').astype(int))
                 .groupby('CLV Tier', observed=True)
                 .agg(customers=('Customer','count'), avg_clv=('Customer Lifetime Value','mean'), response_rate=('response_flag','mean'), avg_claim=('Total Claim Amount','mean')))
value_summary

In [ ]:
clv_p75 = df['Customer Lifetime Value'].quantile(.75)
high_value_nonresponders = df[(df['Customer Lifetime Value'] >= clv_p75) & (df['Response'] == 'No')]
len(high_value_nonresponders), high_value_nonresponders['Customer Lifetime Value'].mean()

## 5. Claims monitoring

In [ ]:
p50, p75, p90 = df['Total Claim Amount'].quantile([.50,.75,.90])
df['Claim Band'] = pd.cut(df['Total Claim Amount'], [-np.inf,p50,p75,p90,np.inf], labels=['Low','Moderate','High','Very High'])
claims = (df.groupby('Claim Band', observed=True)
          .agg(customers=('Customer','count'), avg_claim=('Total Claim Amount','mean'), avg_premium=('Monthly Premium Auto','mean'), avg_clv=('Customer Lifetime Value','mean')))
claims

These claim bands describe the observed sample. They are not a future-risk score and should not be used as a live underwriting rule.

## 6. Marketing-response model

Run `python/insurance_customer_360_analysis.py` for the full preprocessing pipeline and class-weighted logistic-regression model. Evaluate ROC-AUC, average precision, precision/recall and the confusion matrix rather than accuracy alone because the response target is imbalanced.

## 7. Business interpretation

- Validate high-performing offer/channel combinations with controlled testing before rollout.
- Use high-CLV non-responders as a CRM follow-up audience, not as a churn label.
- Review claim-heavy segments with additional exposure and underwriting data before making risk or pricing decisions.
- Track complaints with future campaign and renewal outcomes if longitudinal data becomes available.